In [1]:
import sionna.rt as rt
import numpy as np
import matplotlib.pyplot as plt

relay_scene = rt.load_scene(
    rt.scene.munich,
    merge_shapes=False
)

relay_scene.frequency = 3.5e9

relay_scene.tx_array = rt.PlanarArray(
    num_rows=1,
    num_cols=1,
    pattern="iso",
    polarization = "V"
)

relay_scene.rx_array = relay_scene.tx_array

jitc_llvm_init(): LLVM API initialization failed ..


In [2]:
initial_positions = {
    "base_tx": [495.777, 211.072, 19.704],
    "relay": [225.087, -202.369, 50.0],
    "user_a": [46.182, 84.560, 1.5],
    "user_b": [28.599, 225.094, 1.5],
    "user_c": [-537.994, -398.940, 1.5]
}

base_tx = rt.Transmitter(
    name = "base_tx",
    position=initial_positions["base_tx"],
    power_dbm=30,
    display_radius=4
)

relay_rx = rt.Receiver(
    name = "relay_rx",
    position=initial_positions["relay"],
    display_radius=5
)

relay_tx = rt.Transmitter(
    name = "relay_tx",
    position=initial_positions["relay"],
    power_dbm=20,
    display_radius=3
)

users = {}

for name in ["user_a", "user_b", "user_c"]:
    users[name] = rt.Receiver(
        name = name,
        position=initial_positions[name],
        display_radius=4
    )

devices = [
    base_tx,
    relay_tx,
    relay_rx,
    *users.values()
]

for device in devices:
    if relay_scene.get(device.name) is not None:
        relay_scene.remove(device.name)

    relay_scene.add(devices)

print("Transmitters:", list(relay_scene.transmitters))
print("Receivers:", list(relay_scene.receivers))

relay_scene.preview()

Transmitters: ['base_tx', 'relay_tx']
Receivers: ['relay_rx', 'user_a', 'user_b', 'user_c']


In [3]:
if relay_scene.get("relay_tx") is not None:
    relay_scene.remove("relay_tx")

print("Передавачі:", list(relay_scene.transmitters))
print("Приймачі:", list(relay_scene.receivers))

Передавачі: ['base_tx']
Приймачі: ['relay_rx', 'user_a', 'user_b', 'user_c']


In [4]:
relay_solver = rt.PathSolver()

base_los_paths = relay_solver(
    scene=relay_scene,
    max_depth=0,
    los = True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False
)

In [5]:
base_receiver_names = list(relay_scene.receivers.keys())
valid = np.array(base_los_paths.valid)

for i, name in enumerate(base_receiver_names):
    has_los = bool(valid[i].any())
    print(f"base_tx -> {name}: LOS = {has_los}")

base_tx -> relay_rx: LOS = True
base_tx -> user_a: LOS = False
base_tx -> user_b: LOS = False
base_tx -> user_c: LOS = False


In [6]:
relay_scene.preview(paths=base_los_paths)

In [7]:
base_reflection_paths = relay_solver(
    scene=relay_scene,
    max_depth=2,
    los=True,
    specular_reflection=True,
    diffuse_reflection=False,
    refraction=False
)

In [8]:
valid = np.array(base_reflection_paths.valid)

for i, name in enumerate(base_receiver_names):
    path_count = np.count_nonzero(valid[i])
    print(f"base_tx -> {name}, find paths - {path_count}")

base_tx -> relay_rx, find paths - 2
base_tx -> user_a, find paths - 0
base_tx -> user_b, find paths - 0
base_tx -> user_c, find paths - 0


In [9]:
relay_scene.preview(paths=base_reflection_paths)

In [11]:
for name in ["base_tx", "relay_rx"]:
    if relay_scene.get(name) is not None:
        relay_scene.remove(name)

if relay_scene.get("relay_tx") is None:
    relay_scene.add(relay_tx)

print("Transmitters:", list(relay_scene.transmitters))
print("Receivers:", list(relay_scene.receivers))

Transmitters: ['relay_tx']
Receivers: ['user_a', 'user_b', 'user_c']


In [12]:
relay_los_paths = relay_solver(
    scene=relay_scene,
    max_depth=0,
    los=True,
    specular_reflection=False,
    diffuse_reflection=False,
    refraction=False
)

In [13]:
relay_receiver_names = list(relay_scene.receivers.keys())
valid = np.array(relay_los_paths.valid)

for i, name in enumerate(relay_receiver_names):
    has_los = bool(valid[i].any())
    print(f"relaey_tx -> {name}: LOS = {has_los}")

relaey_tx -> user_a: LOS = False
relaey_tx -> user_b: LOS = False
relaey_tx -> user_c: LOS = False


In [15]:
relay_scene.preview(paths=relay_los_paths)